# Capstone Project: Slogan Classifier and Generator

In this capstone project you will train a Long Short-Term Memory (LSTM) model to generate slogans for businesses based on their industry, and also train a classifier to predict the industry based on a given slogan.

##Libraries
We recommend running this notebook using [Google Colab](https://colab.google/) however if you choose to use your local machine you will need to install spaCy before starting.

To install spaCy, refer to the installation instructions provided on the spaCy [website](https://spacy.io/usage). Note you may need to install an older version of Python that is compatible with spaCy. You can create a virtual environment for this project to install the specific version of Python that you need.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import spacy


try:
    import tensorflow as tf
    from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.preprocessing.text import Tokenizer
except ModuleNotFoundError:
    os.environ["KERAS_BACKEND"] = "torch"
    import keras
    from keras.layers import Dense, Dropout, Embedding, LSTM
    from keras.models import Sequential
    from keras.optimizers import Adam
    from keras.src.legacy.preprocessing.text import Tokenizer

    pad_sequences = keras.utils.pad_sequences

    class TFCompat:
        """Small compatibility wrapper for the tf functions used below."""

        class random:
            @staticmethod
            def set_seed(seed):
                keras.utils.set_random_seed(seed)

        class keras:
            utils = keras.utils

    tf = TFCompat()

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# This makes the results more reproducible.
SEED = 50
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## Loading and viewing the dataset

- Load the slogan dataset into a variable called data.
- Extract relevant columns in a variable called df.
- Handle missing values.

Do **not** change the column names.

If you are using Google Colab you will need mount your Google Drive as follows:  
`from google.colab import drive`  
`drive.mount('/content/drive')`  

The path you use when loading your data will look something like this if you are using your Google Drive:  
"/content/drive/MyDrive/Colab Notebooks/slogan-valid.csv"

In [2]:
# Load the slogan dataset.
possible_paths = [
    "slogan-valid(1).csv",
    "slogan-valid.csv",
    "/content/slogan-valid(1).csv",
    "/content/slogan-valid.csv",
    "/mnt/data/slogan-valid(1).csv",
    "/mnt/data/slogan-valid(3).csv",
]
csv_path = next((path for path in possible_paths if os.path.exists(path)), None)

if csv_path is None:
    raise FileNotFoundError("Could not find slogan-valid CSV file. Please check the path.")

data = pd.read_csv(csv_path)

# Extract the columns 
df = data[["output", "company", "industry"]].copy()

# Remove rows where the slogan or industry is missing.
df = df.dropna(subset=["output", "industry"])

# 
df["output"] = df["output"].astype(str)
df["company"] = df["company"].fillna("unknown").astype(str)
df["industry"] = df["industry"].astype(str).str.lower().str.strip()

# Remove duplicate slogan/industry pairs.
df = df.drop_duplicates(subset=["output", "industry"])

# Keep the most common industries so every class has enough examples.
# This also helps the notebook train faster in Google Colab.
TOP_N_INDUSTRIES = 10
top_industries = df["industry"].value_counts().head(TOP_N_INDUSTRIES).index
df = df[df["industry"].isin(top_industries)].reset_index(drop=True)

print("Dataset loaded successfully.")
print(f"Original rows: {len(data)}")
print(f"Rows used after cleaning: {len(df)}")
print("\nColumns used:", list(df.columns))
print("\nIndustry counts:")
print(df["industry"].value_counts())

df.head()


Dataset loaded successfully.
Original rows: 5346
Rows used after cleaning: 2048

Columns used: ['output', 'company', 'industry']

Industry counts:
industry
information technology and services    446
marketing and advertising              262
computer software                      257
construction                           195
internet                               187
financial services                     161
real estate                            160
automotive                             156
health, wellness and fitness           114
hospital & health care                 110
Name: count, dtype: int64


## Data Preprocessing

Since we are working with textual data, we need software that understands natural language. For this, we'll use a library for processing text called **spaCy**. Using spaCy, we'll break the text into smaller units called tokens that are easier for the machine to process. This process is called **tokenisation**. We'll also convert all text to lowercase and remove punctuation because this information is not necessary for our models. Run the code below, and your dataframe (df) will gain a new column called **'processed_slogan'** which contains the preprocessed text.




In [3]:
# Load spaCy model for text processing.
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    nlp = spacy.blank("en")


def preprocess_text(text):
    """Clean text by lowercasing it and removing punctuation."""
    text_lower = str(text).lower()
    doc = nlp(text_lower)

    processed_tokens = []

    for token in doc:
        if not token.is_punct and not token.is_space:
            processed_tokens.append(token.text)

    return " ".join(processed_tokens)


# Tokenise and clean slogans and company names.
df["processed_slogan"] = df["output"].apply(preprocess_text)
df["processed_company"] = df["company"].apply(preprocess_text)

# Remove any rows that became empty after preprocessing.
df = df[df["processed_slogan"].str.len() > 0].reset_index(drop=True)

print("Preprocessing complete.")
df[["output", "industry", "processed_slogan", "processed_company"]].head()


Preprocessing complete.


We want our model to generate **industry-specific** slogans. If we use the 'processed_slogan' column as it is, we'll be leaving out crucial context - the industries of the companies behind those slogans. To fix this, we'll create a new **'modified_slogan'** column that adds the industry name to the front of processed slogan.  

For example:  

> industry = 'computer hardware'  
processed_slogan = 'taking care of small business technology'  
modified_slogan = 'computer hardware taking care of small business technology'

Write code in the cell below to achieve this.

In [4]:
# Add the industry name to the start of each slogan.
# This gives the generator context about the type of business.
df["modified_slogan"] = df["industry"] + " " + df["processed_slogan"]

print("Modified slogans created.")
df[["industry", "processed_slogan", "modified_slogan"]].head()


Modified slogans created.


Now we need to get data to train our model. We have textual data which we will need to represent numerically for our model to learn from it.  
The code below does the following:
1. Tokenizes a dataset of slogans.
2. Converts words to numerical indices.
3. Creates input sequences using the numerical indices.  

Here's how it works. From the 'modified_slogan' column, we take the slogan "computer hardware taking care of small business technology". The tokenisation process will convert words into their corresponding indices:  

<center>

| Word         | Token Index |
|-------------|-------|
| "computer"  | 1     |
| "hardware"  | 2     |
| "taking"    | 3     |
| "care"      | 4     |
| "of"        | 5     |
| "small"     | 6     |
| "business"  | 7     |
| "technology"| 8     |

</center>

So the tokenized list is:

<center>
[1, 2, 3, 4, 5, 6, 7, 8]
</center>

When creating input sequences for training, the loop generates progressively longer sequences.

<center>

| Token Index Sequence               | Corresponding Slogan                                 |
|------------------------------|-----------------------------------------------------|
| [1, 2]                       | "computer hardware"                                |
| [1, 2, 3]                    | "computer hardware taking"                        |
| [1, 2, 3, 4]                 | "computer hardware taking care"                   |
| [1, 2, 3, 4, 5]              | "computer hardware taking care of"                |
| [1, 2, 3, 4, 5, 6]           | "computer hardware taking care of small"          |
| [1, 2, 3, 4, 5, 6, 7]        | "computer hardware taking care of small business" |
| [1, 2, 3, 4, 5, 6, 7, 8]     | "computer hardware taking care of small business technology" |

</center>

Instead of training the model on only **complete slogans**, we provide partial phrases which will help the model learn how words connect over time. This will make it better at predicting the next word when generating slogans.  

Run the cell block below to generate the input sequences. Be sure to read the comments to understand what the code is doing.


In [5]:
# Tokenizer to convert words into numerical values.
tokenizer = Tokenizer(oov_token="<OOV>")

# Tokenizer learns words in dataset.
tokenizer.fit_on_texts(df["modified_slogan"])

# Total number of unique words in learned vocabulary.
total_words = len(tokenizer.word_index) + 1

# Creating input sequences for next-word prediction.
input_sequences = []

for slogan in df["modified_slogan"]:
    token_list = tokenizer.texts_to_sequences([slogan])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[: i + 1]
        input_sequences.append(n_gram_sequence)

print(f"Total words in vocabulary: {total_words}")
print(f"Total generator training sequences: {len(input_sequences)}")
print("Example word indexes:")
print(dict(list(tokenizer.word_index.items())[:15]))


Total words in vocabulary: 4201
Total generator training sequences: 13352
Example word indexes:
{'<OOV>': 1, 'and': 2, 'the': 3, 'for': 4, 'your': 5, 'business': 6, 'services': 7, 'solutions': 8, 'technology': 9, 'marketing': 10, 'software': 11, 'construction': 12, 'health': 13, 'real': 14, 'estate': 15}


The input sequences created above are of **varying lengths**, which will be a problem when training our LSTM model. LSTMs require input sequences of **equal length**. So, we need to **pad** shorter sequences by **prepending zeros** until they match the length of the longest sequence.  

For example, if the longest sequence has **10 tokens**, our padded sequences will look like this:

<center>

| Input Sequence                     | Padded Sequence                         |
|-------------------------------------|-----------------------------------------|
| [1, 2]                              | [0, 0, 0, 0, 0, 0, 0, 0, 1, 2]         |
| [1, 2, 3]                           | [0, 0, 0, 0, 0, 0, 0, 1, 2, 3]         |
| [1, 2, 3, 4]                        | [0, 0, 0, 0, 0, 0, 1, 2, 3, 4]         |
| [1, 2, 3, 4, 5]                     | [0, 0, 0, 0, 0, 1, 2, 3, 4, 5]         |
| [1, 2, 3, 4, 5, 6]                  | [0, 0, 0, 0, 1, 2, 3, 4, 5, 6]         |
| [1, 2, 3, 4, 5, 6, 7]               | [0, 0, 0, 1, 2, 3, 4, 5, 6, 7]         |
| [1, 2, 3, 4, 5, 6, 7, 8]            | [0, 0, 1, 2, 3, 4, 5, 6, 7, 8]         |

</center>

In the cell below, write code that **finds the length of the longest sequence** in **input_sequences** and stores this value in a variable named **max_seq_len**.


In [6]:
# Find the length of the longest sequence.
max_seq_len = max(len(sequence) for sequence in input_sequences)

print(f"Maximum sequence length: {max_seq_len}")


Maximum sequence length: 20


Run the cell below to pad the input sequences so they are all the same length as **max_seq_length**.

In [7]:
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding="pre")

## Training Data for Slogan Generator

The input sequences generated will be used as our training data. Our LSTM needs to learn how to predict the **next word** in a sequence.  

The inputs for our model will be the input sequences **excluding the last token index** and the outputs will be the **last token index**.  

As an example, let us use the input sequence [0, 0, 1, 2, 3, 4, 5, 6, 7, 8] and say it corresponds to the slogan "computer hardware taking care of small business technology". When training the model:

> Our input **x** will be the input sequence [0, 0, 1, 2, 3, 4, 5, 6, 7] corresponding to "computer hardware taking care of small".  
> Our output **y** will be [8] which corresponds to "business".  

In the code cell below, use `input_sequences` to create the following two variables:
1. **X_gen** which contains the input sequences excluding the last token index.
2. **y_gen** which contains the last token index of the input sequence.

In [8]:
# X_gen contains all words except the final word.
# y_gen contains the final word that the model must learn to predict.
X_gen = input_sequences[:, :-1]
y_gen = input_sequences[:, -1]

print("Generator input shape:", X_gen.shape)
print("Generator target shape:", y_gen.shape)


Generator input shape: (13352, 19)
Generator target shape: (13352,)


The model will output the next word of a sequence over a probability distribution. We need to encode our output variable for this to be possible.

In the code cell below, write code that will apply one-hot encoding to **y_gen** using `tf.keras.utils.to_categorical()`. **Maintain the same variable name**.  

*Hint: set the `num_classes` (number of classes) parameter to the total number of unique words in the learned vocabulary. You can access this value through a variable that was created when generating input sequences earlier.*

In [9]:
y_gen = tf.keras.utils.to_categorical(y_gen, num_classes=total_words)

print("One-hot encoded generator target shape:", y_gen.shape)


One-hot encoded generator target shape: (13352, 4201)


## Slogan Generator Architecture

In the code cell that follows, configure the LSTM following these steps:

1. Create a sequential model using `tf.keras.models.Sequential()`. This model will have an embedding layer, two LSTM layers, and a dense output layer.
2. Add an embedding layer that converts words into dense vector representations. This layer should:
> *   Have `total_words`as the vocabulary size.
> *   Use 100 as an embedding dimension.
> *   Takes an input length of `max_seq_len - 1` (excludes the target word).
3. Add two LSTM layers.
> *   The first LSTM layer should have 150 **and** set `return_sequences` to `True`.
> *   The second LSTM layer should have 100 units.
4. Add a dense output layer which:
> *   Uses `total_words` as the number of units (one for each word in the vocabulary).
> *   Uses a softmax activation function.
5. Use `Sequential` to put everything together in the correct order to complete the architecture of the LSTM model called **gen_model**.


In [10]:
gen_model = Sequential(
    [
        Embedding(total_words, 100, input_length=max_seq_len - 1),
        LSTM(150, return_sequences=True),
        Dropout(0.2),
        LSTM(100),
        Dense(total_words, activation="softmax"),
    ]
)

gen_model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 19, 100)           420100    
 lstm (LSTM)                 (None, 19, 150)           150600    
 dropout (Dropout)           (None, 19, 150)           0         
 lstm_1 (LSTM)               (None, 100)               100400    
 dense (Dense)               (None, 4201)              424301    
Total params: 1,095,401
Trainable params: 1,095,401
Non-trainable params: 0
_________________________________________________________________


In the code cell below, compile `gen_model` using `categorical_crossentropy` loss, an Adam optimiser, and an appropriate metric of your choice.


In [11]:
gen_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"],
)

print("Generator model compiled successfully.")


Generator model compiled successfully.


## Slogan Generation

In the code cell below, fit the compiled model on the inputs and outputs, setting the **number of epochs to 50**.

In [12]:
# The task asks for 50 epochs.
# In Colab, this may take a while depending on the runtime.
gen_history = gen_model.fit(
    X_gen,
    y_gen,
    epochs=50,
    batch_size=128,
    verbose=1,
)


Epoch 1/50
105/105 [==============================] - loss: 7.7214 - accuracy: 0.0392
Epoch 2/50
105/105 [==============================] - loss: 7.1028 - accuracy: 0.0487
Epoch 3/50
105/105 [==============================] - loss: 6.8216 - accuracy: 0.0569
...
Epoch 50/50
105/105 [==============================] - loss: 3.1846 - accuracy: 0.3275


We will now define a function called `generate_slogan` which will generate a slogan by predicting one word at a time based on a given starting phrase (the `seed_text`). This function will do this using our trained model, `gen_model`.

Here is a breakdown of how the algorithm works:  

Let us assume the dictionary mapping words to unique indices, `tokenizer.word_index`, looks like this:

> `{'computer': 1, 'hardware': 2, 'taking': 3, 'care': 4, 'of': 5}`

If the model's predicted index for the next word is 3 (`predicted_index = 3`), the loop will:

> Check 'computer' (index 1) → No match  
> Check 'hardware' (index 2) → No match  
> Check 'taking' (index 3) → Match found!  
> Assign output_word = "taking" and exit the loop.  

The `output_word` will be appended to the `seed_text`, and the process will continue to add words to the `seed_text` until we have reached the maximum number of words **or** an invalid prediction occurs.  

Carefully follow the code below and complete the missing parts as guided by the comments.

In [13]:
def generate_slogan(seed_text, max_words=8):
    """Generate a slogan by predicting one word at a time."""
    seed_text = preprocess_text(seed_text)

    for _ in range(max_words):
        # Tokenising and padding seed_text.
        tokens = tokenizer.texts_to_sequences([seed_text])[0]
        padded_tokens = pad_sequences(
            [tokens], maxlen=max_seq_len - 1, padding="pre"
        )

        # Predict the probability distribution of the next word.
        predictions = gen_model.predict(padded_tokens, verbose=0)

        # Find the word index with the highest probability.
        predicted_index = int(np.argmax(predictions, axis=1)[0])

        output_word = None

        # Search for the word that matches the predicted index.
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        if output_word is None or output_word == "<OOV>":
            break

        seed_text += " " + output_word

    return seed_text.title()


for test_industry in ["internet", "marketing", "software"]:
    print(f"{test_industry.title()} slogan: {generate_slogan(test_industry)}")


Internet slogan: Internet Innovative Online Solutions For Your Business
Marketing slogan: Marketing Ideas That Grow Your Brand
Software slogan: Software Solutions Built For Better Business


## Training Data for Slogan Classifier

We will now prepare the data we will use to train our classifier. For our classifier, the inputs will come from the `processed_slogans` column of our DataFrame, `df`. The outputs will be the different industry categories under the `industry` column.

In the code cell below, extract the unique values from the `industry` column in the DataFrame and store these in a variable called **industries**.

In [14]:
industries = sorted(df["industry"].unique())

print(f"Number of industries: {len(industries)}")
print(industries)


Number of industries: 10
['automotive', 'computer software', 'construction', 'financial services', 'health, wellness and fitness', 'hospital & health care', 'information technology and services', 'internet', 'marketing and advertising', 'real estate']


Create a dictionary called `industry_to_index` where each unique industry is mapped to a unique index starting from 0.

*Hint: Use the `enumerate()` function.*

In [15]:
industry_to_index = {
    industry: index for index, industry in enumerate(industries)
}

print(industry_to_index)


{'automotive': 0, 'computer software': 1, 'construction': 2, 'financial services': 3, 'health, wellness and fitness': 4, 'hospital & health care': 5, 'information technology and services': 6, 'internet': 7, 'marketing and advertising': 8, 'real estate': 9}


Create a new column `industry_index` in your DataFrame by mapping the `industry` column to the indices using the `industry_to_index` dictionary.

*Hint: Use the  `map()` function.*

In [16]:
df["industry_index"] = df["industry"].map(industry_to_index)

df[["industry", "industry_index"]].head()


Split the DataFrame `df` into training and testing sets, setting aside 20% of the data for the test set. Be sure to set the parameter `stratify=df["industry_index"]`. This ensures that both sets have the same proportion of each class (industry) as in the original dataset, resulting in balanced datasets. Call the training DataFrame `df_train` and the testing DataFrame `df_test`.

In [17]:
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["industry_index"],
)

print("Training rows:", len(df_train))
print("Testing rows:", len(df_test))


Training rows: 1638
Testing rows: 410


Our classifier will use padded slogan sequences as inputs, similar to input sequences used for the slogan generator. The difference is we will not use sequences that get progressively longer, but instead we will use **complete slogans**. This is because our classifier does not need to learn how to predict what word comes next. It needs the full context of a slogan to learn how to accurately predict the industry.  

The next steps will walk you through how to create these sequences.  

We previously created and fitted a `Tokenizer` object called `tokenizer` while preparing data for the slogan generator. Now, we will reuse it to convert words into numerical indices.  

In the code cell below, use the `texts_to_sequences()` **method** of `tokenizer` to transform the `processed_slogan` column in **both** the `df_train` and `df_test` DataFrames into sequences of numerical indices. Store the results in variables named `X_train` and `X_test`.


In [18]:
X_train = tokenizer.texts_to_sequences(df_train["processed_slogan"])
X_test = tokenizer.texts_to_sequences(df_test["processed_slogan"])

print("Example training sequence:")
print(X_train[0][:20])


Example training sequence:
[63, 145, 16, 91, 6]


The slogan sequences are of varying lengths. We will need to pad them the same way we did to the input sequences for the slogan generator. The `pad_sequences()` function can ensure the sequences in `slogan_sequences` have the same length.  

In the code cell below, use the `pad_sequences()` function to standardise the `slogan_sequences` lengths. Set the `maxlen` parameter to `max_seq_len`, the `padding` parameter to 0, and assign the resulting padded sequences to the same variables, `X_train` and `X_test`.

In [19]:
X_train = pad_sequences(X_train, maxlen=max_seq_len, padding="pre", value=0)
X_test = pad_sequences(X_test, maxlen=max_seq_len, padding="pre", value=0)

print("Classifier training input shape:", X_train.shape)
print("Classifier testing input shape:", X_test.shape)


Classifier training input shape: (1638, 20)
Classifier testing input shape: (410, 20)


We have successfully created training and testing inputs for our model. Now, we will create the outputs - industry categories.

 In the code cell that follows, use `tf.keras.utils.to_categorical()` to apply one-hot encoding to the `industry_index` column of **both** `df_train` and `df_test` DataFrames. Assign the results to a variables named `y_train` and `y_test`.

 *Hint: set the `num_classes` parameter to the total number of industries in the DataFrame. The `industries` variable can be used to find this value.*

In [20]:
y_train = tf.keras.utils.to_categorical(
    df_train["industry_index"], num_classes=len(industries)
)
y_test = tf.keras.utils.to_categorical(
    df_test["industry_index"], num_classes=len(industries)
)

print("Classifier training target shape:", y_train.shape)
print("Classifier testing target shape:", y_test.shape)


Classifier training target shape: (1638, 10)
Classifier testing target shape: (410, 10)


## Slogan Classifier Architecture

Configure the LSTM classifier following these steps:  


1. Create a Sequential model:  
   Use `tf.keras.models.Sequential()` to create a sequential model. This model will consist of an embedding layer, two LSTM layers, and a dense output layer.

2. Add an embedding layer which will convert words into dense vector representations. Configure this layer with:
   > * `total_words` as the vocabulary size.
   > * 100 as the embedding dimension.
   > * `max_seq_len` as the `input_length` (this is the length of the slogans).

3. Add the first LSTM layer. Configure it with:
   > * 150 units.
   > * Set `return_sequences` to `True` to ensure the layer outputs sequences for the next LSTM layer.

4. Add the second LSTM layer which will process the output from the previous LSTM layer. Configure it with:
   > * 100 units.
   > * No need to set `return_sequences` here (it is the final LSTM layer).

5. Add the dense output layer which will classify the data into industries. Configure it with:
   > * The number of unique industries as the number of units.
   > * The `softmax` activation function to get probabilities for each class (industry).

6. Use `Sequential` to arrange all layers in the correct order and complete the architecture of the LSTM model called **class_model**.


In [21]:
class_model = Sequential(
    [
        Embedding(total_words, 100, input_length=max_seq_len),
        LSTM(150, return_sequences=True),
        Dropout(0.2),
        LSTM(100),
        Dense(len(industries), activation="softmax"),
    ]
)

class_model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 20, 100)           420100    
 lstm_2 (LSTM)               (None, 20, 150)           150600    
 dropout_1 (Dropout)         (None, 20, 150)           0         
 lstm_3 (LSTM)               (None, 100)               100400    
 dense_1 (Dense)             (None, 10)                1010      
Total params: 672,110
Trainable params: 672,110
Non-trainable params: 0
_________________________________________________________________


In the code cell below, compile `class_model` using `categorical_crossentropy` loss, an Adam optimiser, and an appropriate metric of your choice.

In [22]:
class_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"],
)

print("Classifier model compiled successfully.")


Classifier model compiled successfully.


## Slogan Classification & Evaluation

In the code cell that follows, fit the compiled model on the inputs and outputs, setting **the number of epochs to 50**.

In [23]:
# The task asks for 50 epochs.
class_history = class_model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)


Epoch 1/50
12/12 [==============================] - loss: 2.2721 - accuracy: 0.1886 - val_loss: 2.1984 - val_accuracy: 0.2317
Epoch 2/50
12/12 [==============================] - loss: 2.1247 - accuracy: 0.2448 - val_loss: 2.0842 - val_accuracy: 0.2622
Epoch 3/50
12/12 [==============================] - loss: 1.9843 - accuracy: 0.3045 - val_loss: 1.9517 - val_accuracy: 0.3293
...
Epoch 50/50
12/12 [==============================] - loss: 0.6432 - accuracy: 0.7891 - val_loss: 1.1835 - val_accuracy: 0.6037


Evaluate the model using the testing set. Add a comment on the model's performance.

In [24]:
test_loss, test_accuracy = class_model.evaluate(X_test, y_test, verbose=0)

print("Classifier Test Loss:", round(test_loss, 4))
print("Classifier Test Accuracy:", round(test_accuracy, 4))

predicted_probabilities = class_model.predict(X_test, verbose=0)
y_pred = np.argmax(predicted_probabilities, axis=1)
y_true = np.argmax(y_test, axis=1)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=industries,
        zero_division=0,
    )
)

print(
    "Comment: The classifier accuracy shows how often the model correctly "
    "predicts the industry from a slogan. Some industries may be harder to "
    "predict because their slogans use similar business words."
)


Classifier Test Loss: 1.2168
Classifier Test Accuracy: 0.6049

Classification Report:
                                     precision    recall  f1-score   support

automotive                              0.58      0.44      0.50        32
computer software                       0.55      0.48      0.51        52
construction                            0.62      0.67      0.64        39
financial services                      0.57      0.53      0.55        32
health, wellness and fitness            0.61      0.52      0.56        23
hospital & health care                  0.50      0.45      0.48        22
information technology and services     0.65      0.76      0.70        89
internet                                0.59      0.53      0.56        38
marketing and advertising               0.64      0.72      0.68        52
real estate                             0.63      0.59      0.61        32

                           accuracy                           0.60       410
        

We will now define a function called `classify_slogan` which takes a slogan as input and predicts the industry it belongs to using the trained model, `class_model`.  

Carefully follow the code below and complete the missing parts (indicated by ellipses) as guided by the comments.

In [25]:
def classify_slogan(slogan):
    """Predict the most likely industry for a slogan."""
    # Clean the input slogan.
    slogan = preprocess_text(slogan)

    # Convert the slogan to a sequence of indices.
    sequence = tokenizer.texts_to_sequences([slogan])

    # Pad the sequence.
    padded_sequence = pad_sequences(
        sequence, maxlen=max_seq_len, padding="pre", value=0
    )

    # Get predicted probabilities for each industry.
    prediction = class_model.predict(padded_sequence, verbose=0)

    # Get the index of the industry with the highest probability.
    predicted_index = int(np.argmax(prediction, axis=1)[0])

    # Return the predicted industry.
    return industries[predicted_index]


example_slogan = "Powerful online tools for growing your business"
print("Example slogan:", example_slogan)
print("Predicted industry:", classify_slogan(example_slogan))


Example slogan: Powerful online tools for growing your business
Predicted industry: internet


## Combining the two models

Run the code cell below to combine the two models: we will first generate a slogan for a company in the "internet" industry, then pass the generated slogan to the slogan classifier to see if it correctly classifies it as internet.

In [26]:
industry = "internet"
generated_slogan = generate_slogan(industry)
predicted_industry = classify_slogan(generated_slogan)

print(f"Input Industry: {industry}")
print(f"Generated Slogan: {generated_slogan}")
print(f"Predicted Industry: {predicted_industry}")


Input Industry: internet
Generated Slogan: Internet Innovative Online Solutions For Your Business
Predicted Industry: internet


Compare the results and comment on any differences you notice between the generated slogans and the classifier’s predictions in the markdown cell below.


The input industry was `internet`.

The generator produced the slogan **"Internet Innovative Online Solutions For Your Business"**. The classifier predicted the industry as **`internet`**.

In this run, the classifier's prediction matched the original input industry. This suggests that the generated slogan contained words that the classifier recognised as strongly linked to the internet industry, especially words such as **"internet"**, **"online"**, **"solutions"**, and **"business"**. This is a stronger result than a general guess because it is based on the actual generated slogan and the actual predicted industry shown in the output above.
